# IBKR Skew Analysis

Reads option chain snapshots saved by `fetch_ibkr_data.ipynb` and produces:

1. **Skew metrics table** — σATM, RR25, BF25 per expiry for the latest snapshot  
2. **Term structure plots** — how σATM, RR25 and BF25 vary across expiries today  
3. **Vol surface + PDF** — SVI-fitted IV surface and Breeden-Litzenberger risk-neutral PDF  
4. **Historical evolution** — how the skew has changed day-by-day (once multiple snapshots exist)

> **RR25 sign convention (equity):** RR25 = σ_25P − σ_25C  
> Positive = puts more expensive than calls = market pricing downside risk

In [ ]:
# ── 0. Imports ───────────────────────────────────────────────────────────────
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import PchipInterpolator

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from skew.data_store import load_option_snapshots, load_skew_metrics, save_skew_metrics
from skew.utils import (
    bs_delta, bs_price_forward,
    iv_at_target_delta, fit_svi_slice, eval_svi_iv,
)

print("imports OK")

In [ ]:
# ── 1. Config ────────────────────────────────────────────────────────────────
TICKER         = "QQQ"
DB_PATH        = str(ROOT / "data" / "options.db")
RISK_FREE_RATE = 0.0525   # approximate risk-free rate
DIV_YIELD      = 0.0015   # QQQ ~0.15% annual dividend yield
MIN_T_DAYS     = 7        # skip expiries closer than this

In [ ]:
# ── 2. Load + preprocess helpers ─────────────────────────────────────────────

def load_ibkr_data(ticker, db_path=DB_PATH):
    """Load IBKR option snapshots, compute T from snapshot/expiry dates."""
    df = load_option_snapshots(ticker, db_path=db_path)
    if df.empty:
        raise RuntimeError(f"No data for {ticker}. Run fetch_ibkr_data.ipynb first.")

    df = df[df["source"].isin(["ibkr", "ibkr_hist"])].copy()
    if df.empty:
        raise RuntimeError(f"No IBKR-sourced rows for {ticker} (only yfinance data found).")

    df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
    df["expiry"]        = pd.to_datetime(df["expiry"])
    df["T"]             = (df["expiry"] - df["snapshot_date"]).dt.days / 365.0
    df                  = df[df["T"] >= MIN_T_DAYS / 365].copy()

    for col in ["implied_vol", "strike", "delta", "underlying_price"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    df["is_call"] = df["option_type"].str.lower().str.strip().isin(["call", "c"])
    return df.sort_values(["snapshot_date", "expiry", "is_call", "strike"]).reset_index(drop=True)


def _iv_at_delta_ibkr(df_side, target_delta):
    """Interpolate IV at target_delta using IBKR-stored delta values."""
    d = df_side.dropna(subset=["delta", "implied_vol"]).sort_values("delta")
    if len(d) < 2:
        return np.nan
    deltas = d["delta"].values
    ivs    = d["implied_vol"].values
    if deltas.min() <= target_delta <= deltas.max():
        return float(np.interp(target_delta, deltas, ivs))
    return float(ivs[0] if target_delta < deltas.min() else ivs[-1])


def compute_skew_for_snapshot(df_snap, r=RISK_FREE_RATE, q=DIV_YIELD):
    """Compute σATM, RR25, BF25 per expiry for one snapshot."""
    spot = float(df_snap["underlying_price"].median())
    rows = []

    for expiry, g in df_snap.groupby("expiry"):
        T = float(g["T"].median())
        if T < MIN_T_DAYS / 365:
            continue

        F = spot * np.exp((r - q) * T)

        g_iv = g.dropna(subset=["implied_vol"])
        if len(g_iv) < 3:
            continue

        # ATM: closest strike to spot
        sigma_atm = float(g_iv.iloc[(g_iv["strike"] - spot).abs().argsort().iloc[0]]["implied_vol"])

        calls = g_iv[g_iv["is_call"]].sort_values("strike")
        puts  = g_iv[~g_iv["is_call"]].sort_values("strike")

        # Use IBKR delta if available, else compute via BS
        use_ibkr_delta = (
            calls["delta"].notna().sum() >= 2 and
            puts["delta"].notna().sum()  >= 2
        )

        if use_ibkr_delta:
            sigma_25c = _iv_at_delta_ibkr(calls,  0.25)
            sigma_25p = _iv_at_delta_ibkr(puts,  -0.25)
        else:
            for side, is_call in [(calls, True), (puts, False)]:
                side["delta"] = side.apply(
                    lambda row: bs_delta(spot, row["strike"], r, q,
                                         row["implied_vol"], T, is_call), axis=1)
                side["iv"] = side["implied_vol"]
            sigma_25c = iv_at_target_delta(calls,  0.25)
            sigma_25p = iv_at_target_delta(puts,  -0.25)

        if pd.notna(sigma_25c) and pd.notna(sigma_25p):
            rr25 = sigma_25p - sigma_25c   # equity convention: positive = put premium
            bf25 = (sigma_25c + sigma_25p) / 2.0 - sigma_atm
        else:
            rr25 = bf25 = np.nan

        rows.append({
            "expiry":    expiry,
            "T":         round(T, 4),
            "days":      round(T * 365),
            "spot":      round(spot, 2),
            "forward":   round(F, 2),
            "sigma_atm": sigma_atm,
            "sigma_25c": sigma_25c,
            "sigma_25p": sigma_25p,
            "rr25":      rr25,
            "bf25":      bf25,
            "n_strikes": len(g_iv),
        })

    return pd.DataFrame(rows).sort_values("T").reset_index(drop=True)

In [ ]:
# ── 3. Compute & save skew metrics for all IBKR snapshot dates ───────────────
import sqlite3

df_all = load_ibkr_data(TICKER)

snapshot_dates = sorted(df_all["snapshot_date"].dt.date.unique())
print(f"{TICKER} — {len(snapshot_dates)} snapshot date(s)  "
      f"({snapshot_dates[0]}  →  {snapshot_dates[-1]})")

# One-time cleanup: remove any skew_metrics rows whose calc_date doesn't match
# a real snapshot_date (these are stale rows written with today's date by the
# old code that hardcoded calc_date = date.today()).
snap_date_strs = {str(d) for d in snapshot_dates}
try:
    with sqlite3.connect(DB_PATH) as _con:
        bad = _con.execute(
            "SELECT DISTINCT calc_date FROM skew_metrics WHERE ticker=?",
            (TICKER.upper(),)
        ).fetchall()
        bad_dates = {r[0] for r in bad} - snap_date_strs
        if bad_dates:
            print(f"Removing {len(bad_dates)} stale calc_date(s): {sorted(bad_dates)}")
            _con.execute(
                f"DELETE FROM skew_metrics WHERE ticker=? AND calc_date IN "
                f"({','.join('?'*len(bad_dates))})",
                [TICKER.upper()] + sorted(bad_dates)
            )
            _con.commit()
except Exception as _e:
    print(f"Cleanup skipped: {_e}")

# Only compute metrics for dates not already in the skew_metrics table
existing       = load_skew_metrics(TICKER, db_path=DB_PATH)
existing_dates = set(existing["calc_date"].unique()) if not existing.empty else set()
print(f"Already-computed snapshot dates: {sorted(existing_dates)}")

total_new = 0
for snap_dt in snapshot_dates:
    if str(snap_dt) in existing_dates:
        continue
    df_snap = df_all[df_all["snapshot_date"].dt.date == snap_dt].copy()
    metrics = compute_skew_for_snapshot(df_snap)
    if metrics.empty:
        continue
    metrics["expiry"] = metrics["expiry"].dt.date.astype(str)
    n = save_skew_metrics(metrics, TICKER, db_path=DB_PATH, calc_date=str(snap_dt))
    total_new += n
    print(f"  {snap_dt}: {n} new rows")

print(f"New skew rows saved: {total_new}  "
      f"(already computed: {len(existing_dates)} date(s))")

# Load latest snapshot for surface analysis
latest_date = snapshot_dates[-1]
df_latest   = df_all[df_all["snapshot_date"].dt.date == latest_date].copy()
skew_latest = compute_skew_for_snapshot(df_latest)

pct = "{:.1%}"
fmt = {c: pct for c in ["sigma_atm", "sigma_25c", "sigma_25p", "rr25", "bf25"]}
display(
    skew_latest[["expiry", "days", "sigma_atm", "sigma_25c", "sigma_25p",
                 "rr25", "bf25", "n_strikes"]]
    .style
    .format(fmt)
    .bar(subset=["rr25"], align="zero", color=["seagreen", "tomato"])
    .bar(subset=["bf25"], align="zero", color=["steelblue", "steelblue"])
    .set_caption(f"Skew metrics — {TICKER}  ({latest_date})")
    .hide(axis="index")
)

In [ ]:
# ── 4. Term structure — σATM, RR25, BF25 by expiry ──────────────────────────
if skew_latest.empty:
    print("No skew data to plot.")
else:
    x      = skew_latest["days"].values
    xlabs  = [str(e.date() if hasattr(e, "date") else e)
               for e in skew_latest["expiry"]]
    rr_col = ["tomato" if v >= 0 else "seagreen" for v in skew_latest["rr25"]]

    fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
    fig.suptitle(f"{TICKER} — Skew Term Structure  ({latest_date})",
                 fontsize=13, fontweight="bold")

    axes[0].plot(x, skew_latest["sigma_atm"] * 100, "o-",
                 color="steelblue", lw=2, ms=5)
    axes[0].set_ylabel("σ ATM (%)"); axes[0].grid(True, alpha=0.3)
    axes[0].set_title("ATM implied vol — term structure", fontsize=9)

    axes[1].bar(x, skew_latest["rr25"] * 100, color=rr_col, alpha=0.75, width=8)
    axes[1].axhline(0, color="black", lw=0.8)
    axes[1].set_ylabel("RR 25Δ (%)\n(+) = put premium")
    axes[1].grid(True, alpha=0.3, axis="y")
    axes[1].set_title("Risk reversal — positive = puts more expensive (fear / downside hedge)",
                      fontsize=9)

    axes[2].bar(x, skew_latest["bf25"] * 100, color="darkorange", alpha=0.75, width=8)
    axes[2].axhline(0, color="black", lw=0.8)
    axes[2].set_ylabel("BF 25Δ (%)")
    axes[2].set_xlabel("Days to expiry")
    axes[2].grid(True, alpha=0.3, axis="y")
    axes[2].set_title("Butterfly — positive = fat tails priced in (big move expected)",
                      fontsize=9)
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(xlabs, rotation=45, ha="right", fontsize=8)

    plt.tight_layout()
    plt.show()

In [ ]:
# ── 5. Vol surface + Risk-Neutral PDF (SVI + Breeden-Litzenberger) ───────────

def _build_iv_surface(smiles, forwards, maturities, disc_factors,
                      k_pad=0.15, Nk=301, smooth_sigma=3):
    """
    Build an IV surface on a uniform log-moneyness grid.

    k bounds are computed automatically from the actual data range across all
    smiles, extended by k_pad on each side.  This ensures the grid always
    extends well past the last data strike for every expiry, eliminating the
    PDF spike that appeared when the boundary coincided with the data edge.

    Smoothing uses mode='nearest' (not default 'reflect') so the Gaussian
    filter does not create an artificial flat region at the grid boundary.
    """
    M = len(maturities)

    # Compute log-moneyness range from actual data across all smiles
    k_vals = []
    for i in range(M):
        F  = float(forwards[i])
        df = smiles[i].dropna()
        df = df[df["strike"] > 0]
        if len(df):
            k_vals.extend(np.log(df["strike"].values / F).tolist())
    k_lo = (min(k_vals) - k_pad) if k_vals else -0.7
    k_hi = (max(k_vals) + k_pad) if k_vals else  0.7

    k_grid = np.linspace(k_lo, k_hi, Nk)
    Kmat   = np.zeros((M, Nk))
    SIGMA  = np.zeros((M, Nk))

    for i in range(M):
        F, T     = float(forwards[i]), float(maturities[i])
        K_target = F * np.exp(k_grid)
        Kmat[i]  = K_target

        df = smiles[i].dropna()
        df = df[(df["strike"] > 0) & (df["implied_vol"] > 0)].sort_values("strike")
        if len(df) < 2:
            SIGMA[i] = 0.20
            continue

        K_in    = df["strike"].values
        iv_in   = np.clip(df["implied_vol"].values, 1e-4, 5.0)
        iv_eval = None

        # ── SVI: evaluate across full grid, wings extrapolate naturally ───────
        if len(df) >= 5:
            params = fit_svi_slice(K_in, iv_in, F, T)
            if params is not None:
                iv_eval = eval_svi_iv(params, K_target, F, T)

        # ── PCHIP fallback: slope-preserving wing extrapolation ───────────────
        if iv_eval is None:
            pchip   = PchipInterpolator(K_in, iv_in, extrapolate=False)
            iv_eval = pchip(K_target)

            left = k_grid < np.log(K_in[0]  / F)
            if left.any() and len(K_in) >= 2:
                slope = (iv_in[1]  - iv_in[0])  / (K_in[1]  - K_in[0])
                iv_eval[left] = iv_in[0] + slope * (K_target[left] - K_in[0])

            right = k_grid > np.log(K_in[-1] / F)
            if right.any() and len(K_in) >= 2:
                slope = (iv_in[-1] - iv_in[-2]) / (K_in[-1] - K_in[-2])
                iv_eval[right] = iv_in[-1] + slope * (K_target[right] - K_in[-1])

            valid = np.where(np.isfinite(iv_eval))[0]
            if len(valid):
                iv_eval[:valid[0]]    = iv_eval[valid[0]]
                iv_eval[valid[-1]+1:] = iv_eval[valid[-1]]
            else:
                iv_eval = np.full_like(K_target, np.nanmedian(iv_in))

        SIGMA[i] = np.clip(
            gaussian_filter1d(iv_eval.astype(float), smooth_sigma, mode='nearest'), 1e-4, 5.0
        )

    return Kmat, SIGMA


def _price_and_pdf(Kmat, SIGMA, forwards, maturities, disc_factors, smooth=2):
    M, _ = SIGMA.shape
    PDF  = np.zeros_like(SIGMA)
    for i in range(M):
        F, T, D = float(forwards[i]), float(maturities[i]), float(disc_factors[i])
        Cv = np.array([bs_price_forward(True, F, K, s, T, D)
                        for K, s in zip(Kmat[i], SIGMA[i])])
        C  = gaussian_filter1d(Cv.astype(float), smooth, mode='nearest')
        K  = Kmat[i]; h = K[1] - K[0]
        Cpp = np.empty_like(C)
        Cpp[1:-1] = (C[2:] - 2*C[1:-1] + C[:-2]) / (h*h)
        Cpp[0]  = Cpp[1]; Cpp[-1] = Cpp[-2]
        f       = np.clip(Cpp / max(D, 1e-16), 0, None)
        area    = np.trapz(f, K)
        PDF[i]  = f / area if area > 1e-12 else f
    return PDF


# ── Build surface from latest snapshot ───────────────────────────────────────
spot_latest = float(df_latest["underlying_price"].median())
forwards_arr, maturities_arr, disc_arr, smiles = [], [], [], []

for exp in sorted(df_latest["expiry"].unique()):
    g = df_latest[df_latest["expiry"] == exp].dropna(subset=["implied_vol"])
    T = float(g["T"].median())
    F = spot_latest * np.exp((RISK_FREE_RATE - DIV_YIELD) * T)
    D = np.exp(-RISK_FREE_RATE * T)
    otm = pd.concat([
        g[~g["is_call"] & (g["strike"] <  F)][["strike", "implied_vol"]],
        g[ g["is_call"] & (g["strike"] >= F)][["strike", "implied_vol"]],
    ]).drop_duplicates("strike")
    if len(otm) >= 3:
        forwards_arr.append(F); maturities_arr.append(T)
        disc_arr.append(D);     smiles.append(otm)

if len(smiles) < 2:
    print("Need at least 2 expiries for surface. Fetch more data first.")
else:
    Kmat, SIGMA = _build_iv_surface(smiles, forwards_arr, maturities_arr, disc_arr)
    PDF         = _price_and_pdf(Kmat, SIGMA, forwards_arr, maturities_arr, disc_arr)
    grid_T      = np.array(maturities_arr)

    # Restrict plots to the strike range covered by actual data
    all_data_strikes = np.concatenate([s["strike"].values for s in smiles])
    K_plot_lo = all_data_strikes.min() * 0.98
    K_plot_hi = all_data_strikes.max() * 1.02

    # ── Heatmaps ──────────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle(f"{TICKER} — Vol Surface & PDF  ({latest_date})",
                 fontsize=12, fontweight="bold")

    mask_k = (Kmat[0] >= K_plot_lo) & (Kmat[0] <= K_plot_hi)
    im0 = axes[0].imshow(SIGMA[:, mask_k] * 100, origin="lower", aspect="auto",
                          extent=[Kmat[0, mask_k].min(), Kmat[0, mask_k].max(),
                                  grid_T.min(), grid_T.max()])
    plt.colorbar(im0, ax=axes[0]).set_label("IV (%)")
    axes[0].set_xlabel("Strike"); axes[0].set_ylabel("Time to expiry (yr)")
    axes[0].set_title("Implied Volatility Surface (SVI)", fontsize=10)
    axes[0].axvline(spot_latest, color="white", lw=1, ls="--", alpha=0.7)

    im1 = axes[1].imshow(PDF[:, mask_k], origin="lower", aspect="auto",
                          extent=[Kmat[0, mask_k].min(), Kmat[0, mask_k].max(),
                                  grid_T.min(), grid_T.max()])
    plt.colorbar(im1, ax=axes[1]).set_label("Density")
    axes[1].set_xlabel("Strike"); axes[1].set_ylabel("Time to expiry (yr)")
    axes[1].set_title("Risk-Neutral PDF (Breeden-Litzenberger)", fontsize=10)
    axes[1].axvline(spot_latest, color="white", lw=1, ls="--", alpha=0.7)

    plt.tight_layout(); plt.show()

    # ── Per-expiry slices ──────────────────────────────────────────────────────
    n_show = min(6, len(grid_T))
    idx    = np.unique(np.linspace(0, len(grid_T)-1, n_show, dtype=int))
    cmap   = plt.cm.viridis

    fig, (ax_iv, ax_pdf) = plt.subplots(1, 2, figsize=(13, 4))
    for j, i in enumerate(idx):
        col   = cmap(j / max(len(idx)-1, 1))
        label = f"T={grid_T[i]:.2f}y"
        mk    = (Kmat[i] >= K_plot_lo) & (Kmat[i] <= K_plot_hi)
        ax_iv.plot(Kmat[i, mk],  SIGMA[i, mk] * 100, lw=1.5, color=col, label=label)
        ax_pdf.plot(Kmat[i, mk], PDF[i, mk],          lw=1.5, color=col, label=label)

    for ax in (ax_iv, ax_pdf):
        ax.axvline(spot_latest, color="grey", lw=0.8, ls="--", label="spot")
        ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

    ax_iv.set(xlabel="Strike",  ylabel="IV (%)",  title="Vol smile slices")
    ax_pdf.set(xlabel="Strike", ylabel="Density", title="PDF slices")
    plt.tight_layout(); plt.show()

In [ ]:
# ── 6. Historical evolution — σATM, RR25, BF25 over time ────────────────────
hist = load_skew_metrics(TICKER, db_path=DB_PATH)

if hist.empty:
    print("No historical skew data yet. Run daily to accumulate.")
else:
    hist["calc_date"] = pd.to_datetime(hist["calc_date"])
    hist["expiry"]    = pd.to_datetime(hist["expiry"])

    dates    = sorted(hist["calc_date"].unique())
    expiries = sorted(hist["expiry"].unique())
    print(f"{len(dates)} observation date(s)  ×  {len(expiries)} expiry/ies")

    if len(dates) < 2:
        print("Only 1 date — need multiple snapshots for a time series.\n"
              "Run fetch_ibkr_data.ipynb daily to build up history.")
    else:
        cmap   = plt.cm.plasma
        colors = [cmap(i / max(len(expiries)-1, 1)) for i in range(len(expiries))]

        fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
        fig.suptitle(
            f"{TICKER} — Skew History by Expiry  "
            f"({dates[0].date()} → {dates[-1].date()})",
            fontsize=12, fontweight="bold",
        )

        for exp, col in zip(expiries, colors):
            h   = hist[hist["expiry"] == exp].sort_values("calc_date")
            lbl = str(exp.date())
            axes[0].plot(h["calc_date"], h["sigma_atm"] * 100,
                         "o-", color=col, ms=3, lw=1.5, label=lbl)
            axes[1].plot(h["calc_date"], h["rr25"] * 100,
                         "o-", color=col, ms=3, lw=1.5, label=lbl)
            axes[2].plot(h["calc_date"], h["bf25"] * 100,
                         "o-", color=col, ms=3, lw=1.5, label=lbl)

        axes[0].set_ylabel("σ ATM (%)"); axes[0].grid(True, alpha=0.3)
        axes[1].set_ylabel("RR 25Δ (%)")
        axes[1].axhline(0, color="k", lw=0.7, ls="--"); axes[1].grid(True, alpha=0.3)
        axes[2].set_ylabel("BF 25Δ (%)")
        axes[2].axhline(0, color="k", lw=0.7, ls="--"); axes[2].grid(True, alpha=0.3)
        axes[2].set_xlabel("Date")
        axes[2].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))

        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, title="Expiry", loc="center right",
                   bbox_to_anchor=(1.0, 0.5), fontsize=7, framealpha=0.8)
        plt.tight_layout(rect=[0, 0, 0.88, 1])
        plt.show()

        # ── RR25 heatmap: expiry × date ────────────────────────────────────────
        pivot = hist.pivot_table(
            index="expiry", columns="calc_date", values="rr25", aggfunc="mean"
        )
        if pivot.shape[0] > 1 and pivot.shape[1] > 1:
            fig, ax = plt.subplots(
                figsize=(max(6, pivot.shape[1] * 0.7), max(3, pivot.shape[0] * 0.5))
            )
            vmax = np.nanpercentile(np.abs(pivot.values), 95)
            im   = ax.imshow(pivot.values * 100, aspect="auto",
                             cmap="RdYlGn_r", origin="lower",
                             vmin=-vmax*100, vmax=vmax*100)
            plt.colorbar(im, ax=ax).set_label("RR 25Δ (%)")
            ax.set_yticks(range(len(pivot.index)))
            ax.set_yticklabels([str(e.date()) for e in pivot.index], fontsize=8)
            ax.set_xticks(range(len(pivot.columns)))
            ax.set_xticklabels([str(d.date()) for d in pivot.columns],
                               rotation=45, ha="right", fontsize=8)
            ax.set_title(f"{TICKER} — RR 25Δ (%) by expiry × date  "
                         "(red = put premium / fear)", fontsize=10)
            plt.tight_layout(); plt.show()